In [33]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, accuracy_score

In [34]:
df = pd.read_csv("Cleaned_EMP.csv")
df.head()

,Unnamed: 0,EmployeeID,Name,Age,Department,Salary,JoinDate,City,PerformanceScore,Bonus,Experience,Gender,TenureYears,TCompension,PStatus,Seniority,AttritionRisk,PromotionStatus
0,0,101,Ali,25.0,4,50000.0,2021-03-15,1,7.5,5000.0,2,1,5,55000.0,Good,Junior,0.0,No Promotion
1,1,102,Sara,29.0,1,60000.0,2020-07-10,2,8.2,0.0,5,0,6,60000.0,Good,Junior,0.0,Review
2,2,103,Ahmed,36.0,2,75000.0,2019-01-05,1,9.1,7000.0,6,1,7,82000.0,Excellent,Junior,0.0,Review
3,3,104,Ayesha,27.0,0,68000.0,2022-05-20,0,8.5,6000.0,3,0,4,74000.0,Good,Junior,0.0,Review
4,4,105,Usman,31.0,4,52000.0,2018-09-17,1,7.0,0.0,7,1,8,52000.0,Average,Mid,1.0,No Promotion


In [35]:
df = df.drop(columns=["Unnamed: 0",'EmployeeID','JoinDate','TCompension','PromotionStatus','Name'])
df.head()

,Age,Department,Salary,City,PerformanceScore,Bonus,Experience,Gender,TenureYears,PStatus,Seniority,AttritionRisk
0,25.0,4,50000.0,1,7.5,5000.0,2,1,5,Good,Junior,0.0
1,29.0,1,60000.0,2,8.2,0.0,5,0,6,Good,Junior,0.0
2,36.0,2,75000.0,1,9.1,7000.0,6,1,7,Excellent,Junior,0.0
3,27.0,0,68000.0,0,8.5,6000.0,3,0,4,Good,Junior,0.0
4,31.0,4,52000.0,1,7.0,0.0,7,1,8,Average,Mid,1.0


In [36]:
target_reg = "Salary"
target_clf = "AttritionRisk"

In [37]:
X = df.drop(columns=[target_reg,target_clf])
yr = df[target_reg]
yc = df[target_clf].astype(int)  

In [38]:
categorical_cols = ["Department", "City", "Gender", "PStatus", "Seniority"]
numerical_cols = [col for col in X.columns if col not in categorical_cols]

In [39]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numerical_cols)
    ]
)


In [40]:
reg_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42))
])

In [50]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, yr, test_size=0.2, random_state=43
)
preprocessor.fit(X_train_r)

ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['Department', 'City', 'Gender', 'PStatus',
                                  'Seniority']),
                                ('num', 'passthrough',
                                 ['Age', 'PerformanceScore', 'Bonus',
                                  'Experience', 'TenureYears'])])

In [51]:
reg_model.fit(X_train_r, y_train_r)
pred_r = reg_model.predict(X_test_r)

In [59]:
print("SALARY MODEL (Regression)")
print("R2 Score:", r2_score(y_test_r, pred_r))

SALARY MODEL (Regression)
R2 Score: 0.9972547717941992


In [60]:
cv_reg = cross_val_score(reg_model, X, yr, cv=5, scoring="r2")
print("CV Mean R2:", cv_reg.mean())

CV Mean R2: 0.9918795383993965


In [61]:
clf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42))
])

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, yc, test_size=0.2, random_state=42
)

clf_model.fit(X_train_c, y_train_c)
pred_c = clf_model.predict(X_test_c)

print("\n📊 ATTRITION MODEL (Classification)")
print("Accuracy:", accuracy_score(y_test_c, pred_c))

cv_clf = cross_val_score(clf_model, X, yc, cv=5, scoring="accuracy")
print("CV Mean Accuracy:", cv_clf.mean())


📊 ATTRITION MODEL (Classification)
Accuracy: 0.9897959183673469
CV Mean Accuracy: 0.9897959183673469


In [62]:
new_employee = {
    "Age": 51,
    "Department": 1,
    "City": 1,
    "PerformanceScore": 6.6,
    "Bonus": 10000,
    "Experience": 10,
    "Gender": 1,
    "TenureYears": 7,
    "PStatus": "Good",
    "Seniority": "Mid"
}

input_df = pd.DataFrame([new_employee])
salary_pred = reg_model.predict(input_df)[0]
attrition_pred = clf_model.predict(input_df)[0]
attrition_prob = clf_model.predict_proba(input_df)[0]
print("\n==================== PREDICTION ====================")
print("💰 Predicted Salary:", round(salary_pred, 2))

print("🚪 Attrition Risk:", "Yes" if attrition_pred == 1 else "No")
print("📊 Probability:", attrition_prob)


==================== PREDICTION ====================
💰 Predicted Salary: 78425.0
🚪 Attrition Risk: No
📊 Probability: [0.825 0.175]
